# Lab 4 — Build the Audience, Then Buy It
Name: Kuksa Zahar | Group: KM-43

## Setup

In [1]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str, keep_default_na=False)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

In [2]:
OCT = pd.date_range('2024-10-01', '2024-10-31')
BIG = ['CBS', 'NBC', 'ABC', 'FOX', 'FNC']
programs['bdate'] = (programs['start_ts'] - pd.Timedelta(hours=6)).dt.normalize()
programs['duration'] = (programs['end_ts'] - programs['start_ts']).dt.total_seconds() / 60
daily_weights['broadcast_date'] = pd.to_datetime(daily_weights['broadcast_date'])
dw = daily_weights[daily_weights['resp_type'].isin(['R', 'L'])].copy()
va = viewer_attributes.drop_duplicates(['respondent_id', 'attr_id'])
age = va[va['attr_id'] == 300].set_index('respondent_id')['value']
person_ids = va.loc[va['attr_id'].isin([101, 102]), 'respondent_id']
adult_ids = age.index[age.index.isin(person_ids) & (age >= 18)]
sex = va[va['attr_id'].isin([201, 202])].set_index('respondent_id')['attr_id'].map({201: 'Men', 202: 'Women'})

def size(ids):
    return (dw[dw['respondent_id'].isin(ids)].groupby('broadcast_date')['weight']
            .sum().reindex(OCT, fill_value=0).mean())

## Task 1.0 — Clean the survey file

In [3]:
s = survey.copy()
answers = survey_layout[survey_layout['RECTYPE'] == 'B'].set_index('KEYWORD')
s['id'] = pd.to_numeric(s['RESPID'].str.strip(), errors='coerce')
s['key'] = s['KEYWORD'].str.strip().str.upper()
s['yes'] = s['VALUE'].str.strip().str.upper().isin(['1', '1.0', 'Y', 'YES', 'TRUE'])
valid = s['id'].isin(adult_ids) & s['key'].isin(answers.index) & s['yes']
x = s.loc[valid, ['id', 'key']].drop_duplicates()
pair_codes = {'X710101': 1, 'X710102': 2, 'X710103': 3}
pc = x[x['key'].isin(pair_codes)].copy()
conflicts = pc.groupby('id').size().gt(1)
pc['pairs'] = pc['key'].map(pair_codes)
pc = pc.sort_values('pairs').drop_duplicates('id', keep='last')
x = pd.concat([x[~x['key'].isin(pair_codes)], pc[['id', 'key']]])
buyers = x['id'].unique()
missing_any = np.setdiff1d(buyers, x.loc[x['key'] == 'X710100', 'id'])
x = pd.concat([x, pd.DataFrame({'id': missing_any, 'key': 'X710100'})])
people = pd.DataFrame(index=adult_ids)
people['buyer'] = people.index.isin(buyers)
people['pairs'] = people.index.map(pc.set_index('id')['pairs']).fillna(0).map({0:'0', 1:'1', 2:'2', 3:'3+'})
for brand, key in {'STRIDE':'X710201', 'KESTREL':'X710202', 'EASYWALK':'X710203'}.items():
    people[brand] = people.index.isin(x.loc[x['key'] == key, 'id'])

audit = pd.DataFrame([
    ['ID spaces, leading zeros or .0', (s['id'].notna() & s['RESPID'].ne(s['id'].fillna(0).astype('int64').astype(str))).sum(), 'Normalize integer ID'],
    ['Keyword case / spaces', s['KEYWORD'].ne(s['key']).sum(), 'Strip and uppercase'],
    ['Alternative yes spellings', (s['yes'] & s['VALUE'].ne('1')).sum(), 'Normalize to yes'],
    ['Header / trailer, invalid ID', s['id'].isna().sum(), 'Drop'],
    ['Not an adult R/L panel member', (s['id'].notna() & ~s['id'].isin(adult_ids)).sum(), 'Drop'],
    ['Unknown answer code', (~s['key'].isin(answers.index)).sum(), 'Drop'],
    ['No / blank answers', (~s['yes']).sum(), 'Drop'],
    ['Duplicate person-keyword', s.loc[valid].duplicated(['id','key']).sum(), 'Keep one'],
    ['Conflicting pair-count rows', conflicts.sum() * 2, 'Keep higher count'],
    ['Missing Any answers', len(missing_any), 'Infer from pairs / brand']
], columns=['issue', 'rows', 'action'])
check = answers[['Text', 'SAMPLE']].copy()
check['SAMPLE'] = pd.to_numeric(check['SAMPLE'])
check['our_count'] = x.groupby('key')['id'].nunique().reindex(check.index, fill_value=0)
assert check['our_count'].eq(check['SAMPLE']).all()
print("Survey data rows:", len(survey))
print("Clean yes answers:", len(x))
print("Adult panel members:", len(people))
print("Non-buyers:", int((~people['buyer']).sum()))
print("People with two pair counts:", int(conflicts.sum()))
print(audit.to_string(index=False))
print(check.to_string())

Survey data rows: 18349
Clean yes answers: 15635
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113
                         issue  rows                   action
ID spaces, leading zeros or .0  1455     Normalize integer ID
         Keyword case / spaces  1104      Strip and uppercase
     Alternative yes spellings  1194         Normalize to yes
  Header / trailer, invalid ID     3                     Drop
 Not an adult R/L panel member   423                     Drop
           Unknown answer code   741                     Drop
            No / blank answers   745                     Drop
      Duplicate person-keyword   933                 Keep one
   Conflicting pair-count rows   226        Keep higher count
           Missing Any answers   238 Infer from pairs / brand
                    Text  SAMPLE  our_count
KEYWORD                                    
X710100              Any    4779       4779
X710101           1 pair    1810       1810
X710102          

**Answer:** I fixed ID/code formatting, removed invalid IDs, unknown codes, non-yes rows and duplicates, kept the higher pair count, and inferred Any from a pair count or brand. Audit counts can overlap; all cleaned counts match SAMPLE. Raw exact-keyword rows give 516 Kestrel answers instead of 354 people (+162, or 45.8%). Raw Any rows give a buyer share of 0.572 versus 0.597 after cleaning, 2.5 percentage points too low (unweighted adult roster).

## Task 1.1 — Four definitions

In [4]:
AUD={'Adults 18+':adult_ids,
     'Shoe buyers':people.index[people['buyer']],
     'Heavy buyers':people.index[people['pairs']=='3+'],
     'Kestrel buyers':people.index[people['KESTREL']],
     'Adults 25-54':adult_ids[age.reindex(adult_ids).between(25,54)]}
AUD['P2+']=pd.Index(dw['respondent_id'].unique())
sizes={a:size(ids) for a,ids in AUD.items()}
print("Adults 18+ (M):",f"{sizes['Adults 18+']/1e6:.1f}")
print("Shoe buyers (M):",f"{sizes['Shoe buyers']/1e6:.1f}")
print("Heavy buyers, 3+ pairs (M):",f"{sizes['Heavy buyers']/1e6:.1f}")
print("Kestrel buyers (M):",f"{sizes['Kestrel buyers']/1e6:.1f}")
print("Adults 25-54 (M):",f"{sizes['Adults 25-54']/1e6:.1f}")
print("Kestrel buyers in panel (n):",len(AUD['Kestrel buyers']))
# Daily audience denominators also serve the telecast rating calculation.
den={a:dw[dw['respondent_id'].isin(ids)].groupby('broadcast_date')['weight'].sum().reindex(OCT,fill_value=0) for a,ids in AUD.items()}

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [5]:
groups={'18-34':adult_ids[age.reindex(adult_ids).between(18,34)],
        '35-54':adult_ids[age.reindex(adult_ids).between(35,54)],
        '55+':adult_ids[age.reindex(adult_ids).ge(55)],
        'Men':adult_ids[sex.reindex(adult_ids).eq('Men')],
        'Women':adult_ids[sex.reindex(adult_ids).eq('Women')]}
profile_rows=[]
for audience in ['Adults 18+','Shoe buyers','Heavy buyers','Kestrel buyers','Adults 25-54']:
    for group,ids in groups.items():
        share=size(np.intersect1d(AUD[audience],ids))/sizes[audience]
        base=size(ids)/sizes['Adults 18+']
        profile_rows.append([audience,group,share,share/base*100])
profile=pd.DataFrame(profile_rows,columns=['audience','group','share','index_vs_adults'])
print(profile.set_index(['audience','group']).round({'share':3,'index_vs_adults':1}).to_string())

                      share  index_vs_adults
audience       group                        
Adults 18+     18-34  0.288            100.0
               35-54  0.319            100.0
               55+    0.393            100.0
               Men    0.488            100.0
               Women  0.512            100.0
Shoe buyers    18-34  0.281             97.5
               35-54  0.336            105.3
               55+    0.383             97.5
               Men    0.485             99.3
               Women  0.515            100.7
Heavy buyers   18-34  0.250             87.0
               35-54  0.365            114.3
               55+    0.385             97.9
               Men    0.438             89.8
               Women  0.562            109.8
Kestrel buyers 18-34  0.299            104.0
               35-54  0.454            142.2
               55+    0.247             62.8
               Men    0.433             88.8
               Women  0.567            110.7
Adults 25-

**Answer:** I would use heavy buyers because they match the brief and have more panel support than Kestrel buyers. Any-shoe buyers are common and have nearly the same demographic mix as all adults; A25–54 is only a proxy for buying. Fused purchases cannot tell us whether watching a program causes purchases, since the link is created through demographics. The task names three age bands and two sex groups, so the table shows those five groups.

## Task 2.1 — Programs by audience

In [6]:
prime = programs[programs['daypart'] == 'Prime'].copy()
sessions = viewings[viewings['resp_type'].isin(['R', 'L'])]
rows = []
for r in prime.itertuples():
    v = sessions[(sessions['network_code'] == r.network_code) &
                 (sessions['start_ts'] < r.end_ts) & (sessions['end_ts'] > r.start_ts)]
    minutes = (v['end_ts'].clip(upper=r.end_ts) - v['start_ts'].clip(lower=r.start_ts)).dt.total_seconds() / 60
    minutes = minutes.groupby(v['respondent_id']).sum()
    w = dw[dw['broadcast_date'] == r.bdate].set_index('respondent_id')['weight']
    wm = minutes * w.reindex(minutes.index).fillna(0)
    row = {'telecast_id': r.telecast_id}
    for a in ['Adults 18+', 'Heavy buyers', 'Kestrel buyers', 'P2+']:
        viewers = wm[wm.index.isin(AUD[a])].sum() / r.duration
        row[a + '_viewers'] = viewers
        row[a + '_rating'] = viewers / den[a].loc[r.bdate] * 100
    rows.append(row)
metrics = prime.merge(pd.DataFrame(rows), on='telecast_id')

rc = rate_card.copy()
rc['program_name'] = rc['program_name'].fillna('')
pp = rc[rc['program_name'] != ''].set_index(['network_code', 'program_name'])['cost_30s_usd']
tr = rc[rc['program_name'] == ''].copy()
def bmin(t):
    h, m = str(t).split(':')
    return int(h) * 60 + int(m) - 360
tr['lo'], tr['hi'] = tr['start_time'].map(bmin), tr['end_time'].map(bmin)
def cost30(r):
    if (r.network_code, r.program_name) in pp.index:
        return pp.loc[(r.network_code, r.program_name)]
    minute = (r.start_ts - r.bdate).total_seconds() / 60 - 360
    match = tr[(tr['network_code'] == r.network_code) & tr['weekdays'].map(lambda d: r.bdate.weekday() in d) &
               (tr['lo'] <= minute) & (tr['hi'] > minute)]
    return match['cost_30s_usd'].item()
metrics['cost_30s'] = [cost30(r) for r in metrics.itertuples()]

WANTED = [('NFL Football','NBC'), ('NFL Football','ABC'), ('Dancing with the Stars','ABC'),
          ('NCIS','CBS'), ('Hannity','FNC'), ('Unsellable Houses','HGTV'),
          ('Halloween Wars','FOOD'), ('Grotesquerie','FX')]
rows = []
for program, network in WANTED:
    m = metrics[(metrics['program_name'] == program) & (metrics['network_code'] == network)].mean(numeric_only=True)
    av, tv, cost = m['Adults 18+_viewers'], m['Heavy buyers_viewers'], m['cost_30s']
    ar, rating = m['Adults 18+_rating'], m['Heavy buyers_rating']
    rows.append([f'{program} ({network})', ar, rating, tv/1000, tv/av, rating/ar*100,
                 cost, cost/av*1000, cost/tv*1000, m['Kestrel buyers_rating']/ar*100, av, tv])
program_table = pd.DataFrame(rows, columns=['program','adults_rating','target_rating','target_viewers_K',
    'composition','index','cost_30s','adults_CPM','target_CPM','kestrel_index','adult_viewers','target_viewers']).set_index('program')
print(program_table.iloc[:, :8].round({'adults_rating':2, 'target_rating':2, 'target_viewers_K':1,
    'composition':3, 'index':1, 'cost_30s':0, 'adults_CPM':0, 'target_CPM':0}).to_string())
print("Target universe (M):", round(sizes['Heavy buyers']/1e6, 1))

                              adults_rating  target_rating  target_viewers_K  composition  index  cost_30s  adults_CPM  target_CPM
program                                                                                                                           
NFL Football (NBC)                     5.82           5.79            2865.0        0.195   99.3  487200.0        33.0       170.0
NFL Football (ABC)                     8.93           8.36            4149.1        0.184   93.6  723600.0        32.0       174.0
Dancing with the Stars (ABC)           3.25           3.14            1564.2        0.191   96.8  158600.0        19.0       101.0
NCIS (CBS)                             2.21           2.43            1206.3        0.216  110.1  178800.0        32.0       148.0
Hannity (FNC)                          1.09           0.84             416.9        0.152   77.1   39900.0        15.0        96.0
Unsellable Houses (HGTV)               0.50           0.64             317.2       

## Task 2.2 — Index, volume or price?

In [7]:
highest_index=program_table['index'].idxmax()
most_viewers=program_table['target_viewers'].idxmax()
cheapest=program_table['target_CPM'].idxmin()
print("Highest index (program):",highest_index)
print("Most target viewers (program):",most_viewers)
print("Lowest target CPM (program):",cheapest)

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): NFL Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** Unsellable Houses (HGTV) has both the highest index and the lowest target CPM, while NFL Football (ABC) delivers the most target viewers. Index measures concentration, not volume or price. I would show all three: buying only the highest index can miss cheap target impressions and the scale of a larger program.

## Task 2.3 — Kestrel buyers

In [8]:
print(program_table[['index','kestrel_index']].round(1).to_string())

                              index  kestrel_index
program                                           
NFL Football (NBC)             99.3           98.5
NFL Football (ABC)             93.6           85.9
Dancing with the Stars (ABC)   96.8           82.7
NCIS (CBS)                    110.1           56.5
Hannity (FNC)                  77.1           42.6
Unsellable Houses (HGTV)      126.8           76.9
Halloween Wars (FOOD)         116.2           95.2
Grotesquerie (FX)              95.9          129.6


**Answer:** Kestrel has only 354 panel members versus 1,661 heavy buyers, and even fewer watch each telecast, so a few weighted people can move its index sharply. I would trust the heavy-buyer index more, while checking viewer counts, household-bootstrap intervals and consistency across weeks. Neither index removes the limitations of demographic data fusion.

## Task 3.1 — Weights

In [9]:
ip_n = hh_attributes.groupby('hashed_ip')['household_id'].nunique()
one = hh_attributes[hh_attributes['hashed_ip'].map(ip_n) == 1].drop_duplicates('hashed_ip')
matched = acr_sessions[['tv_id','hashed_ip']].drop_duplicates().merge(one, on='hashed_ip')
matched['head_age'] = np.where(matched['head_age_band'] == '65+', '65+', 'Under 65')
matched = matched.merge(universe, on=['head_age','income_band'])
acr = acr_sessions[['tv_id','network_code','session_type','start_utc','end_utc']].copy()
for old, new in [('start_utc','start_ts'), ('end_utc','end_ts')]:
    acr[new] = acr[old].dt.tz_convert('America/New_York').dt.tz_localize(None)
acr = acr[(acr['end_ts'] - acr['start_ts']).dt.total_seconds() >= 60]
acr['bdate'] = (acr['start_ts'] - pd.Timedelta(hours=6)).dt.normalize()
last = (acr['end_ts'] - pd.Timedelta(microseconds=1, hours=6)).dt.normalize()
cross = acr.loc[last > acr['bdate'], ['tv_id']].assign(bdate=last)
td = pd.concat([acr[['tv_id','bdate']], cross]).drop_duplicates().merge(matched, on='tv_id')
td['weight'] = td['tv_households'] / td.groupby(['bdate','head_age','income_band'])['tv_id'].transform('size')
weight_sums = td.groupby('bdate')['weight'].sum()
assert np.allclose(weight_sums, universe['tv_households'].sum())
live = acr[acr['session_type'] == 'live']
rows = []
for r in reference_ratings.merge(programs, on='telecast_id').itertuples():
    v = live[(live['network_code'] == r.network_code) & (live['start_ts'] < r.end_ts) & (live['end_ts'] > r.start_ts)]
    overlap = (v['end_ts'].clip(upper=r.end_ts) - v['start_ts'].clip(lower=r.start_ts)).dt.total_seconds()
    day = td[td['bdate'] == r.bdate]
    watched = day['tv_id'].isin(v.loc[overlap >= 60, 'tv_id'])
    rows.append([f'{r.program_name} ({r.network_code})', watched.mean()*100,
                 day.loc[watched,'weight'].sum()/day['weight'].sum()*100, r.hh_rating])
tv_ratings = pd.DataFrame(rows, columns=['program','unweighted','weighted','reference']).set_index('program')
gap_u = (tv_ratings['unweighted'] - tv_ratings['reference']).abs().mean()
gap_w = (tv_ratings['weighted'] - tv_ratings['reference']).abs().mean()
print("TVs matched to one household:", len(matched))
print("Weights sum per day (M):", (weight_sums/1e6).round(1).to_string())
print(tv_ratings.round(2).to_string())
print("Mean gap unweighted (points):", round(gap_u, 2))
print("Mean gap weighted (points):", round(gap_w, 2))
print("Share of gap closed:", round((gap_u-gap_w)/gap_u, 3))

TVs matched to one household: 19607
Weights sum per day (M): bdate
2024-10-14    125.8
2024-10-15    125.8
2024-10-16    125.8
2024-10-17    125.8
2024-10-18    125.8
2024-10-19    125.8
2024-10-20    125.8
2024-10-21    125.8
2024-10-22    125.8
2024-10-23    125.8
2024-10-24    125.8
2024-10-25    125.8
2024-10-26    125.8
2024-10-27    125.8
                                              unweighted  weighted  reference
program                                                                      
MLB Postseason (FOX)                               17.03     17.76      25.86
NFL Football (NBC)                                 22.62     23.75      33.14
NCIS (CBS)                                          5.50      5.80      10.56
Hannity (FNC)                                       1.04      1.35       2.30
Unsellable Houses (HGTV)                            2.12      2.08       3.69
Halloween Baking Championship (FOOD)                0.95      0.90       1.63
Grotesquerie (FX)            

## Task 3.2 — What weighting cannot fix

**Answer:** Weighting raises sports, NCIS and Hannity, while HGTV, FOOD, FX and Bravo move slightly down. Restoring older households helps older-skewing audiences, but closes only 7.5% of the mean absolute rating gap (3.73 to 3.44 points). Income alone has little effect because viewing here is driven more by age than income. Weights cannot recover other sets, people, unrecognized viewing or missing sessions, and still assume matched homes represent unmatched and opted-out homes within each cell.

## Task 4.1 — October predicts October

In [10]:
history=metrics[metrics['bdate'].between('2024-10-01','2024-10-21') & metrics['network_code'].isin(BIG)]
future=metrics[metrics['bdate'].between('2024-10-22','2024-10-28') & metrics['network_code'].isin(BIG)].copy()
keys=['network_code','program_name']
means=history.groupby(keys)[['P2+_rating','Heavy buyers_rating']].mean().rename(columns={'P2+_rating':'forecast_P2','Heavy buyers_rating':'forecast_heavy'})
forecasts=future.merge(means,on=keys,how='inner',validate='many_to_one')
forecasts['program']=forecasts['program_name']+' ('+forecasts['network_code']+')'
forecasts['error_P2']=((forecasts['P2+_rating']-forecasts['forecast_P2']).abs()/forecasts['P2+_rating']).where(forecasts['P2+_rating']>0)
forecasts['error_heavy']=((forecasts['Heavy buyers_rating']-forecasts['forecast_heavy']).abs()/forecasts['Heavy buyers_rating']).where(forecasts['Heavy buyers_rating']>0)
mape_p2=forecasts['error_P2'].mean()*100
mape_heavy=forecasts['error_heavy'].mean()*100
worst=forecasts.loc[forecasts['error_P2'].idxmax()]
print("Telecasts forecast (n):",len(forecasts))
print("MAPE, P2+ (%):",f'{mape_p2:.1f}')
print("MAPE, heavy buyers (%):",f'{mape_heavy:.1f}')
print("Biggest miss, P2+ (program, date):",(worst['program'],worst['bdate'].strftime('%Y-%m-%d')))
print('Positive-actual telecasts scored, P2+:',int(forecasts['error_P2'].notna().sum()))
print('Positive-actual telecasts scored, heavy buyers:',int(forecasts['error_heavy'].notna().sum()))

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): ('Krapopolis (FOX)', '2024-10-27')
Positive-actual telecasts scored, P2+: 101
Positive-actual telecasts scored, heavy buyers: 101


## Task 4.2 — Why it missed

In [11]:
top5 = forecasts.nlargest(5, 'error_P2').copy()
top5['date'] = top5['bdate'].dt.strftime('%Y-%m-%d')
top5['error_%'] = top5['error_P2'] * 100
print(top5[['program','date','forecast_P2','P2+_rating','error_%']].round(2).to_string(index=False))
print('\nSchedule around the biggest misses:')
for r in top5.itertuples():
    schedule = programs[(programs['network_code'] == r.network_code) & (programs['bdate'] == r.bdate) &
                        (programs['end_ts'] >= r.start_ts - pd.Timedelta(hours=3)) & (programs['start_ts'] <= r.start_ts)]
    print(schedule[['program_name','start_ts','end_ts','is_repeat']].to_string(index=False))

                   program       date  forecast_P2  P2+_rating  error_%
          Krapopolis (FOX) 2024-10-27         1.06        0.22   387.76
Universal Basic Guys (FOX) 2024-10-27         0.68        0.26   162.17
        Fire Country (CBS) 2024-10-27         1.70        0.67   154.76
             Accused (FOX) 2024-10-22         0.95        0.37   154.63
  What Would You Do? (ABC) 2024-10-23         0.44        0.18   138.13

Schedule around the biggest misses:
                  program_name            start_ts              end_ts  is_repeat
                  NFL Football 2024-10-27 16:05:00 2024-10-27 19:15:00      False
Local / Syndicated Programming 2024-10-27 19:15:00 2024-10-27 20:00:00      False
                  The Simpsons 2024-10-27 20:00:00 2024-10-27 20:30:00      False
          Universal Basic Guys 2024-10-27 20:30:00 2024-10-27 21:00:00      False
                 Bob's Burgers 2024-10-27 21:00:00 2024-10-27 21:30:00      False
                    Krapopolis 2024-10-

**Answer:** On October 27, FOX's Krapopolis and Universal Basic Guys aired against NBC's Dallas–San Francisco game; the history mixes slots, lead-ins and repeats, and Universal Basic Guys moved from 20:00 on October 20 to 20:30. Fire Country (CBS) was a 22:44 repeat fragment after an NFL-displaced Sunday schedule, unlike its usual Friday hour. Accused (FOX), October 22, stayed in its normal slot, so the schedule does not prove an overrun: episode variation and panel noise are possible. What Would You Do? (ABC), October 23, followed Abbott Elementary, but its short history also includes a Monday after Dancing with the Stars. Heavy buyers have fewer panelists and noisier ratings; small actuals inflate MAPE (38.6% versus 35.8% for P2+). A better forecast would use slot/repeat/lead-in information and episode or competing-event features, tested on held-out weeks.

## Task 5.1 — The plan

In [12]:
spots = {'NFL Football (ABC)':1, 'Unsellable Houses (HGTV)':4, 'Halloween Wars (FOOD)':4,
         'Dancing with the Stars (ABC)':1, 'Grotesquerie (FX)':3}
plan = program_table.loc[list(spots)].copy()
plan['spots'] = pd.Series(spots)
plan['cost'] = plan['spots'] * plan['cost_30s']
plan['target_impressions'] = plan['spots'] * plan['target_viewers']
plan['target_impressions_M'] = plan['target_impressions'] / 1e6
plan_cost, plan_imps = plan['cost'].sum(), plan['target_impressions'].sum()
non_target_cost = (plan['cost'] * (1-plan['composition'])).sum()
assert plan_cost <= 1_000_000
print(plan[['spots','cost','target_impressions_M','composition']].round({'cost':0,'target_impressions_M':1,'composition':3}).to_string())
print("Plan cost ($):", int(round(plan_cost)))
print("Plan target impressions (M):", round(plan_imps/1e6, 1))
print('Unspent budget ($):', int(round(1_000_000-plan_cost)))
print('Cost apportioned to non-target adults ($):', int(round(non_target_cost)))

                              spots      cost  target_impressions_M  composition
program                                                                         
NFL Football (ABC)                1  723600.0                   4.1        0.184
Unsellable Houses (HGTV)          4   51600.0                   1.3        0.249
Halloween Wars (FOOD)             4   38800.0                   0.6        0.228
Dancing with the Stars (ABC)      1  158600.0                   1.6        0.191
Grotesquerie (FX)                 3   22776.0                   0.4        0.188
Plan cost ($): 995376
Plan target impressions (M): 8.0
Unspent budget ($): 4624
Cost apportioned to non-target adults ($): 806042


## Task 5.2 — Recommendation

**Answer:** Buy 1 spot in NFL Football (ABC), 4 in Unsellable Houses (HGTV), 4 in Halloween Wars (FOOD), 1 in Dancing with the Stars (ABC) and 3 in Grotesquerie (FX). The plan costs $995,376, leaves $4,624, and projects 8.0 million gross heavy-buyer impressions at a target CPM of about $124. Football adds volume, while the cheaper programs improve efficiency and diversify the buy. Heavy-buyer backtest MAPE is 38.6%, so this is uncertain delivery, not a guarantee or a prediction interval. Only 19.7% of adult impressions are on target; allocating cost by each show's adult composition associates about $806,042 (81.0%) with non-target adults, excluding children. Confirm November availability, especially seasonal shows, and negotiate make-goods for any guaranteed delivery shortfall. After the week, check actual spots, spend, target impressions/CPM, reach and frequency against the forecast and guarantee.

## Bonus

In [13]:
print("Kaggle username:","Not supplied — competition bonus not attempted")

Kaggle username: Not supplied — competition bonus not attempted


**Answer:** The optional private competition was not entered; no Kaggle username or competition files were supplied. No bonus score is claimed.